In [8]:
import pandas as pd

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [4]:
df = pd.read_csv("data/processed/clean_data.csv")

print(df.shape)
df.head()

(12000, 3)


,clean_text,score,bank
0,dikit verivikasi ribet mohon di perbaiki,1,BCAMobile_reviews
1,sangat terbantu sexali dg adanya m bangking in,1,BCAMobile_reviews
2,kenapa sy tidak bisa mendownlod aplikasi bca,1,BCAMobile_reviews
3,sangat kecewa saya melakukan transfer keterang...,1,BCAMobile_reviews
4,update an baru ada bugs,1,BCAMobile_reviews


In [5]:
texts = df["clean_text"].astype(str).tolist()

# IndoBERT

In [6]:
model = SentenceTransformer(
  "indobenchmark/indobert-base-p1",
  device="cuda" # using rtx 3060
)

c:\Users\PC\anaconda3\envs\skripsi\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\PC\.cache\huggingface\hub\models--indobenchmark--indobert-base-p1. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 13701.03it/s]


In [7]:
embeddings = model.encode(texts,batch_size=32,show_progress_bar=True)

Batches: 100%|██████████| 375/375 [00:13<00:00, 27.20it/s]


In [ ]:
# sanity check
sim = cosine_similarity([embeddings[0]], embeddings[:10])

print(sim)

# expected result
# nilai mendekati 1 = mirip
# lebih kecil = beda

[[0.9999999  0.47826347 0.5376282  0.5520282  0.64250004 0.501776
  0.5075089  0.44884416 0.5895494  0.5600138 ]]


| Nilai similarity | Arti                |
| ---------------- | ------------------- |
| ~0.9             | hampir identik      |
| 0.6 – 0.8        | mirip               |
| 0.4 – 0.6        | agak mirip (normal) |
| <0.3             | beda                |


In [14]:
print(texts[0])
print(texts[1])

dikit verivikasi ribet mohon di perbaiki
sangat terbantu sexali dg adanya m bangking in


# BERTopic

In [16]:
from bertopic import BERTopic

In [17]:
topic_model = BERTopic()

topics, probs = topic_model.fit_transform(texts, embeddings)

In [18]:
df["topic"] = topics
df.head()

,clean_text,score,bank,topic
0,dikit verivikasi ribet mohon di perbaiki,1,BCAMobile_reviews,8
1,sangat terbantu sexali dg adanya m bangking in,1,BCAMobile_reviews,-1
2,kenapa sy tidak bisa mendownlod aplikasi bca,1,BCAMobile_reviews,-1
3,sangat kecewa saya melakukan transfer keterang...,1,BCAMobile_reviews,-1
4,update an baru ada bugs,1,BCAMobile_reviews,-1


In [19]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,7832,-1_di_bisa_saya_nya,"[di, bisa, saya, nya, ini, mau, aplikasi, tida...",[kenapa sekarang setiap ganti kartu di hp apli...
1,0,416,0_kenapa_dibuka_bisa_buka,"[kenapa, dibuka, bisa, buka, ya, kok, knp, tid...","[kenapa aplikasi brimo tidak bisa di buka, gak..."
2,1,181,1_sy_bs_tdk_hrs,"[sy, bs, tdk, hrs, gk, lg, sdh, tp, yg, jg]",[lampu indikator merah selm hari pdhl internet...
3,2,149,2_wajah_verifikasi_gagal_susah,"[wajah, verifikasi, gagal, susah, verivikasi, ...","[gagal verifikasi wajah mulu, verifikasi wajah..."
4,3,130,3_buka_bisa_tidak_bs,"[buka, bisa, tidak, bs, di, dibuka, akses, sus...","[tidak bisa buka lagi, sering tidak bisa di bu..."
...,...,...,...,...,...
94,93,12,93_susah_login_sekarang_referral,"[susah, login, sekarang, referral, to, loginny...","[susah login ulang, sekarang lebih susah login..."
95,94,11,94_dulu_dlu_coba_buntang,"[dulu, dlu, coba, buntang, xixixi, tess, tes, ...","[d coba dulu, coba dulu, bintang aja dulu coba]"
96,95,11,95_ttp_gabisa_sidik_gk,"[ttp, gabisa, sidik, gk, udh, apl, ni, mental,...",[per hari ini tgl tetap tidak bisa sdh melakuk...
97,96,10,96_gag_saldo_mutasi_ribu,"[gag, saldo, mutasi, ribu, gtu, kepotong, uang...",[maling sy trf ke brizzi ga masuk tp saldo kep...


# Evaluating

In [21]:
df["topic"].value_counts().head()

topic
-1    7832
 0     416
 1     181
 2     149
 3     130
Name: count, dtype: int64